# Week 2 — The model is just a rule you can read

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ShayanAhmed12/flyrank-internship-ml/blob/main/notebooks/02_your_first_readable_model.ipynb?flush_cache=true)

You'll:
1. Write a **1-line hand rule** and rank pages with it.
2. Fit a **depth-2 decision tree** and `print` it — the model "learned" a readable if/else. Then compare — where does it beat your rule, and where doesn’t it?
3. See **why you never feed the outcome back in** — that's leakage.

The payoff isn't a high score. It's: *my intuition was rough, the model found the real signal, and I can read exactly what it found.*

## 0. Setup (Colab or local)

In [1]:
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/flyrank-bih/flyrank-ml-internship-starter"
REPO_DIR = "flyrank-ml-internship-starter"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
elif os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")

import pandas as pd, numpy as np
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

# The label: a page is 'declining' when its recent trend is down. Simple, honest starter label.
df["is_declining_label"] = df["trend_direction"].str.lower().eq("down").astype(int)
print(df.shape[0], "pages |  declining rate:", round(df["is_declining_label"].mean(), 3))

30000 pages |  declining rate: 0.542


## 1. A rule you write by hand: `stale x visible`
Intuition: a page worth reviewing is one that is **stale** (not updated in a while) **and** still **visible** (getting impressions). Rank those by how much exposure they have.

In [2]:
stale   = (df["days_since_last_update"] >= 180).astype(int)
visible = (df["impressions_90d"] >= 500).astype(int)
df["hand_rule_score"] = stale * visible * df["impressions_90d"]

top10 = df.sort_values("hand_rule_score", ascending=False).head(10)
top10[["impressions_90d", "days_since_last_update", "avg_position", "ctr", "trend_direction"]]

,impressions_90d,days_since_last_update,avg_position,ctr,trend_direction
16751,61678,194,19.7,0.15,down
16514,59472,194,24.8,0.13,down
7021,25715,194,22.2,0.23,down
21268,13299,193,10.5,0.49,down
11489,7812,194,39.0,0.01,down
12045,7558,193,17.9,0.20,down
698,4590,194,31.0,0.00,down
5327,4556,194,16.4,0.33,down
26810,4429,194,25.3,0.38,down
20837,1697,193,15.8,0.12,down


We need a way to score any ranking. **Precision@K** = of the top K pages a ranking flags, what fraction are actually declining.

In [3]:
def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    topk = np.asarray(labels)[order[:k]]
    return topk.mean()

y = df["is_declining_label"].values
for k in (20, 50):
    print(f"Hand rule  Precision@{k}: {precision_at_k(df['hand_rule_score'], y, k):.3f}")

Hand rule  Precision@20: 0.900
Hand rule  Precision@50: 0.620


## 2. Let a model learn the rule — then read it
A **depth-2 decision tree** can only ask 3 yes/no questions. That constraint is the point: whatever it learns, you can read.

We give it a few **pre-decision** signals — never product flags.

In [4]:
from sklearn.tree import DecisionTreeClassifier, export_text

features = ["content_age_days", "days_since_last_update", "impressions_90d",
            "avg_position", "ctr", "word_count"]
X = df[features].replace([np.inf, -np.inf], np.nan).fillna(0)

tree = DecisionTreeClassifier(max_depth=2, class_weight="balanced", random_state=42)
tree.fit(X, y)

print(export_text(tree, feature_names=features))

|--- impressions_90d <= 5.50
|   |--- avg_position <= 0.75
|   |   |--- class: 0
|   |--- avg_position >  0.75
|   |   |--- class: 0
|--- impressions_90d >  5.50
|   |--- content_age_days <= 312.50
|   |   |--- class: 1
|   |--- content_age_days >  312.50
|   |   |--- class: 0



That printout **is** the model — a human-readable if/else. Now rank pages by the tree's probability and score it the same way.

In [5]:
tree_score = tree.predict_proba(X)[:, 1]
for k in (20, 50):
    hr = precision_at_k(df["hand_rule_score"], y, k)
    tr = precision_at_k(tree_score, y, k)
    print(f"Precision@{k}:  hand rule {hr:.3f}   vs   tree {tr:.3f}")

Precision@20:  hand rule 0.900   vs   tree 0.800
Precision@50:  hand rule 0.620   vs   tree 0.720


Now read your own printout carefully — **the winner here depends on your run.** A depth-2 tree can only give four different scores (one per leaf), so the "top 50" is mostly one big block of tied pages, and different library versions break those ties differently. On some stacks the tree wins at Precision@50; on others the hand rule holds both. **Both results are real.** The stable lesson: a sharp human rule can be excellent at the very top of the list; a model's advantage — when it shows up — appears deeper, where simple rules run out of signal; and any comparison built on heavily tied scores is fragile. Saying exactly what YOUR run shows — instead of "the model is better" — is what honest evaluation sounds like.

## 3. Why you can't feed the outcome back in
Your label is `trend_direction == "down"`, and `trend_pct` is the exact percentage change that bucket is computed from — so it **is** the answer in disguise. Watch what happens if you feed it in as a feature:

In [6]:
X_leaky = df[features + ["trend_pct"]].replace([np.inf, -np.inf], np.nan).fillna(0)
leaky = DecisionTreeClassifier(max_depth=2, class_weight="balanced", random_state=42).fit(X_leaky, y)
print(f"'Leaky' tree Precision@50: {precision_at_k(leaky.predict_proba(X_leaky)[:,1], y, 50):.3f}  <- looks amazing")
print(export_text(leaky, feature_names=features + ["trend_pct"]))

'Leaky' tree Precision@50: 1.000  <- looks amazing
|--- trend_pct <= -20.05
|   |--- word_count <= 212.00
|   |   |--- class: 1
|   |--- word_count >  212.00
|   |   |--- class: 1
|--- trend_pct >  -20.05
|   |--- trend_pct <= -19.95
|   |   |--- class: 0
|   |--- trend_pct >  -19.95
|   |   |--- class: 0



The tree just split on `trend_pct` and nailed the label — because the label is **derived from** `trend_pct`. That's **leakage**: the feature is the answer in disguise, and it teaches you nothing.

That's also why the starter data ships **only observable signals** — the product's own decision flags (health scores, "needs CTR fix", and so on) aren't included, so you can't accidentally train on them. You build from what was knowable *before* the outcome.

> Rule of thumb: if a feature would only be known *because someone already made the decision you're predicting*, it leaks. Leave it out.

## 4. 🔧 Your turn
- Change `max_depth` to 3 or 4 — does Precision@50 improve? Can you still read the tree?
- Swap in different features (drop `impressions_90d`, add `engagement_rate`). What does the tree choose to split on first?
- **Important caveat:** we scored *in-sample* here for teaching. The real pipeline uses **client-holdout** validation (`scripts/03_train_model.py`) so a client's pages never appear in both train and test. Re-run your comparison with a train/test split and see if the gap holds.

Write your experiment below.

In [7]:
# Your experiment here
# Three experiments, all scored with the same precision_at_k as above.

def fit_tree(X_train, y_train, depth):
    return DecisionTreeClassifier(max_depth=depth, class_weight="balanced", random_state=42).fit(X_train, y_train)

# --- Experiment 1: deeper trees (in-sample, same as the teaching cells) ---
print("Experiment 1 - tree depth, scored in-sample")
for depth in (2, 3, 4):
    t = fit_tree(X, y, depth)
    s = t.predict_proba(X)[:, 1]
    print(f"  depth {depth}: P@20 {precision_at_k(s, y, 20):.3f} | P@50 {precision_at_k(s, y, 50):.3f} "
          f"| leaves {t.get_n_leaves()} | distinct scores {len(np.unique(s))}")
print()
print("Depth-3 tree, printed:")
print(export_text(fit_tree(X, y, 3), feature_names=features))


Experiment 1 - tree depth, scored in-sample


  depth 2: P@20 0.800 | P@50 0.720 | leaves 4 | distinct scores 4


  depth 3: P@20 0.550 | P@50 0.680 | leaves 8 | distinct scores 8


  depth 4: P@20 0.700 | P@50 0.660 | leaves 16 | distinct scores 16

Depth-3 tree, printed:


|--- impressions_90d <= 5.50
|   |--- avg_position <= 0.75
|   |   |--- impressions_90d <= 3.50
|   |   |   |--- class: 0
|   |   |--- impressions_90d >  3.50
|   |   |   |--- class: 0
|   |--- avg_position >  0.75
|   |   |--- content_age_days <= 108.50
|   |   |   |--- class: 0
|   |   |--- content_age_days >  108.50
|   |   |   |--- class: 0
|--- impressions_90d >  5.50
|   |--- content_age_days <= 312.50
|   |   |--- ctr <= 0.33
|   |   |   |--- class: 1
|   |   |--- ctr >  0.33
|   |   |   |--- class: 1
|   |--- content_age_days >  312.50
|   |   |--- avg_position <= 25.15
|   |   |   |--- class: 0
|   |   |--- avg_position >  25.15
|   |   |   |--- class: 0



In [8]:
# --- Experiment 2: swap features (drop impressions_90d, add engagement_rate) ---
features_swap = ["content_age_days", "days_since_last_update", "avg_position", "ctr", "word_count", "engagement_rate"]
X_swap = df[features_swap].replace([np.inf, -np.inf], np.nan).fillna(0)
t_swap = fit_tree(X_swap, y, 2)
print("Experiment 2 - depth-2 tree without impressions_90d, with engagement_rate")
print(export_text(t_swap, feature_names=features_swap))
s_swap = t_swap.predict_proba(X_swap)[:, 1]
print(f"In-sample P@20 {precision_at_k(s_swap, y, 20):.3f} | P@50 {precision_at_k(s_swap, y, 50):.3f}")


Experiment 2 - depth-2 tree without impressions_90d, with engagement_rate
|--- avg_position <= 0.55
|   |--- avg_position <= 0.15
|   |   |--- class: 0
|   |--- avg_position >  0.15
|   |   |--- class: 0
|--- avg_position >  0.55
|   |--- content_age_days <= 287.50
|   |   |--- class: 1
|   |--- content_age_days >  287.50
|   |   |--- class: 0

In-sample P@20 0.450 | P@50 0.560


In [9]:
# --- Experiment 3: the important caveat - client-holdout train/test split ---
# Same idea as scripts/03_train_model.py: hold out ~20% of CLIENTS, so no client's pages are in both train and test.
# The hand rule has fixed thresholds (nothing to train), so it is simply scored on the same test rows.
def client_holdout(frame, seed, test_share=0.2):
    clients = frame["client_id"].unique()
    shuffled = np.random.default_rng(seed).permutation(clients)
    test_clients = set(shuffled[: max(1, round(len(shuffled) * test_share))])
    test = frame["client_id"].isin(test_clients).to_numpy()
    return ~test, test

hand = df["hand_rule_score"].values
train, test = client_holdout(df, seed=42)
print(f"Split seed 42: {train.sum():,} train pages / {test.sum():,} test pages "
      f"from {df.loc[test, 'client_id'].nunique()} unseen clients | test declining rate {y[test].mean():.3f}")
for depth in (2, 3, 4):
    t = fit_tree(X[train], y[train], depth)
    s = t.predict_proba(X[test])[:, 1]
    print(f"  depth {depth}: tree P@20 {precision_at_k(s, y[test], 20):.3f}, P@50 {precision_at_k(s, y[test], 50):.3f}   |   "
          f"hand rule P@20 {precision_at_k(hand[test], y[test], 20):.3f}, P@50 {precision_at_k(hand[test], y[test], 50):.3f}")

# One split of 6 clients is noisy, so repeat with 10 different client splits (depth 2, as in the teaching cells).
rows = []
for seed in range(10):
    train, test = client_holdout(df, seed)
    t = fit_tree(X[train], y[train], 2)
    rows.append({
        "seed": seed,
        "test_pages": int(test.sum()),
        "base_rate": y[test].mean(),
        "tree_P@50": precision_at_k(t.predict_proba(X[test])[:, 1], y[test], 50),
        "hand_P@50": precision_at_k(hand[test], y[test], 50),
    })
holdout = pd.DataFrame(rows)
print()
print("10 client-holdout splits, depth-2 tree:")
print(holdout.round(3).to_string(index=False))
print()
print(f"Mean P@50: tree {holdout['tree_P@50'].mean():.3f} vs hand rule {holdout['hand_P@50'].mean():.3f} "
      f"(base rate {holdout['base_rate'].mean():.3f}); tree ahead on {(holdout['tree_P@50'] > holdout['hand_P@50']).sum()}/10 splits")


Split seed 42: 27,675 train pages / 2,325 test pages from 6 unseen clients | test declining rate 0.391


  depth 2: tree P@20 0.600, P@50 0.500   |   hand rule P@20 0.450, P@50 0.440


  depth 3: tree P@20 0.550, P@50 0.520   |   hand rule P@20 0.450, P@50 0.440


  depth 4: tree P@20 0.550, P@50 0.600   |   hand rule P@20 0.450, P@50 0.440



10 client-holdout splits, depth-2 tree:
 seed  test_pages  base_rate  tree_P@50  hand_P@50
    0        5588      0.687       0.90       0.80
    1       10515      0.463       0.48       0.36
    2        3237      0.474       0.64       0.54
    3        8376      0.516       0.52       0.56
    4        3895      0.468       0.38       0.36
    5        2914      0.519       0.76       0.66
    6        8808      0.614       0.66       0.68
    7        4280      0.611       0.78       0.66
    8        4152      0.516       0.62       0.50
    9        3034      0.639       0.72       0.68

Mean P@50: tree 0.646 vs hand rule 0.580 (base rate 0.551); tree ahead on 8/10 splits


### What I found

**1. Deeper trees did *not* help Precision@50 here (in-sample).** Depth 2 gave P@50 0.72, depth 3 gave 0.68 and depth 4 gave 0.66. The depth-3 tree is still readable, but its extra splits (`ctr <= 0.33`, `avg_position <= 25.15`, ...) end in leaves with the *same* class on both sides, so they only nudge probabilities. With 4-16 distinct scores, the top 50 is mostly a block of tied pages, which makes P@20/P@50 jumpy. More depth is not automatically more signal.

**2. Without `impressions_90d`, the tree splits on `avg_position` first.** The split is at `avg_position <= 0.55`, which is really the "position = 0 means no data" flag from the data dictionary, i.e. "does this page get search visibility at all". It never used `engagement_rate`. P@50 dropped to 0.56, roughly the base rate (0.54). So almost all of the depth-2 tree's signal was "is the page visible" plus "is it younger than ~300 days".

**3. With a client-holdout split, the gap mostly holds, but it is smaller and noisy.**
- On the seed-42 split (6 unseen clients, 2,325 pages, 39% declining) the depth-2 tree got P@50 **0.50** against **0.44** for the hand rule. At P@20 the tree also led (0.60 vs 0.45). In-sample the hand rule had *won* P@20, so that ranking flipped.
- Over 10 different client splits, the depth-2 tree averaged P@50 **0.65** against **0.58** for the hand rule (base rate 0.55). The tree was ahead on **8/10** splits, but individual splits ranged from 0.38 to 0.90, so one split can tell either story.

**Takeaway:** the tree's edge over the hand rule seems to survive on unseen clients, but it is a few points, not a blowout, and how much depends a lot on which clients land in the test set. Both rankings are only modestly better than picking pages at random. Reporting the spread over several client splits is more honest than one number.


### Save your work
**Colab:** *File → Save a copy in GitHub* (your submission) and *File → Save a copy in Drive*.

You now have the two core reflexes of applied ML: **discover before you model**, and **prefer a model you can read and can't fool**. That's the whole foundation the capstone builds on.